In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.



Credentials retrieved successfully for prod db.


### Helper Functions

In [4]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

### Push heuristic data

In [ ]:
df = pd.read_excel('/data/aman_singh/acuuracy_check/All_combination_Nov25_live_with_festivals((Autorecovered-312166393861403186)).xlsb', sheet_name = 'Base')
df

,key,month_date,pred_prophet,pred_rf,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,...,RF_Recency NS Heuristic Val,RF_Final Heursitic Val,Remark,Missing,ALL Planning Principle,NON ALL Planning Principle,Skip basis PP,Heuristic > 2x Model,Diff,Comparison
0,BCE1_D231_718589,45991,5.264959,2.7,0.000237,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
1,BCE1_D231_718589,46022,4.547336,3.6,0.000205,0.000162,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
2,BCE1_D231_718589,46053,7.053803,2.7,0.000317,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
3,BCE1_D231_718589,46081,5.662848,1.8,0.000255,0.000081,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
4,BCE1_D231_718589,46112,0.000000,2.7,0.000000,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,1,0.0,P3M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCW2_D463_810125,46022,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164366,QCW2_D463_810125,46053,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164367,QCW2_D463_810125,46081,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164368,QCW2_D463_810125,46112,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M


In [ ]:
df['month_date'] = (
    pd.to_datetime(df['month_date'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
df['run_month'] = (
    pd.to_datetime(df['run_month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)

df['month_date'] = (
    pd.to_datetime(df['month_date']) + pd.offsets.MonthEnd(0)
)
df['run_month'] = (
    pd.to_datetime(df['run_month']) + pd.offsets.MonthEnd(0)
)

In [ ]:
df['run_month'].unique()

<DatetimeArray>
['2025-11-30 00:00:00']
Length: 1, dtype: datetime64[ns]

In [ ]:
df.columns

Index(['key', 'month_date', 'pred_prophet', 'pred_rf', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov',
       'run_month', 'M month', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2', 'Growth Flag',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'LY', 'LLY', 'LY value', 'LLY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'ASM', 'Depot', 'PSKU', 'class', '

In [ ]:
df['month_date'] = df['month_date'].astype(str)
df['run_month'] = df['run_month'].astype(str)
df.columns

Index(['key', 'month_date', 'pred_prophet', 'pred_rf', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov',
       'run_month', 'M month', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2', 'Growth Flag',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'LY', 'LLY', 'LY value', 'LLY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'ASM', 'Depot', 'PSKU', 'class', '

In [ ]:
upload_df = df[['channel','portfolio', 'brand_code', 'class',
        'run_month', 'M month','month_date', 'ASM', 'Depot', 'PSKU','pred_prophet', 'pred_rf','Final Heuristic 2 Vol',
        'RF_Final Heuristic Vol']].rename(
            columns = {'brand_code':'brand', 'class':'Brand Class', 'month_date':'month','pred_prophet':'prophet vol',
                       'pred_rf':'rf_vol', 'Final Heuristic 2 Vol':'prophet heuristic vol',
                       'RF_Final Heuristic Vol':'rf heuristic vol'}
        )
upload_df

,channel,portfolio,brand,Brand Class,run_month,M month,month,ASM,Depot,PSKU,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_HEURISTICS_OUTPUT",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 164370,
 [('rtthbeires/file0.txt',
   'LOADED',
   164370,
   164370,
   1,
   0,
   None,
   None,
   None,
   None)])

### Push shared output

In [46]:
df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Stat Demand Forecast MT_as_on_11th_May_2026.xlsb',
    sheet_name='Base'
)

In [47]:
df['run_month'] = '2026-05-31'
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,NPD Flag,run_month
0,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46203,BCE1,D231,...,0.000347,0.000347,0.000517,0.000231,0.000231,0.000347,0.000593,False,NON-NPD,2026-05-31
1,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46234,BCE1,D231,...,0.000347,0.000347,0.000517,0.000231,0.000289,0.000347,0.000417,False,NON-NPD,2026-05-31
2,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46265,BCE1,D231,...,0.000694,0.000347,0.000517,0.000231,0.000289,0.000347,0.001111,False,NON-NPD,2026-05-31
3,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,46295,BCE1,D231,...,0.000000,0.000347,0.000517,0.000231,0.000405,0.000463,0.000000,False,NON-NPD,2026-05-31
4,MT,B2B,CNO,PCNO(R),A,349274.001420,BCE1_D231_718312,46203,BCE1,D231,...,0.004191,0.002328,0.002212,0.000000,0.001164,0.000000,0.001929,False,NON-NPD,2026-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26883,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811219,46295,MCW2,D463,...,0.000000,0.000541,0.000000,0.000000,0.000000,0.000000,0.000541,False,NON-NPD,2026-05-31
26884,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,46203,MCW2,D463,...,0.000000,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31
26885,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,46234,MCW2,D463,...,0.000000,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31
26886,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,46265,MCW2,D463,...,0.000000,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31


In [48]:
df['Month'] = (
    pd.to_datetime(df['Month'], unit='D', origin='1899-12-30')
      .dt.to_period('M')
      .dt.to_timestamp()
)
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,LY Val (Cr),P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,NPD Flag,run_month
0,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-06-01,BCE1,D231,...,0.000347,0.000347,0.000517,0.000231,0.000231,0.000347,0.000593,False,NON-NPD,2026-05-31
1,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-07-01,BCE1,D231,...,0.000347,0.000347,0.000517,0.000231,0.000289,0.000347,0.000417,False,NON-NPD,2026-05-31
2,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-08-01,BCE1,D231,...,0.000694,0.000347,0.000517,0.000231,0.000289,0.000347,0.001111,False,NON-NPD,2026-05-31
3,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-09-01,BCE1,D231,...,0.000000,0.000347,0.000517,0.000231,0.000405,0.000463,0.000000,False,NON-NPD,2026-05-31
4,MT,B2B,CNO,PCNO(R),A,349274.001420,BCE1_D231_718312,2026-06-01,BCE1,D231,...,0.004191,0.002328,0.002212,0.000000,0.001164,0.000000,0.001929,False,NON-NPD,2026-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26883,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811219,2026-09-01,MCW2,D463,...,0.000000,0.000541,0.000000,0.000000,0.000000,0.000000,0.000541,False,NON-NPD,2026-05-31
26884,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-06-01,MCW2,D463,...,0.000000,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31
26885,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-07-01,MCW2,D463,...,0.000000,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31
26886,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-08-01,MCW2,D463,...,0.000000,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31


In [49]:
df.columns

Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?', 'NPD Flag', 'run_month'],
      dtype='object')

In [ ]:
#df.drop(['Qtr Index Rate', 'Brand Class', 'LY (ROUM)', 'LY Value (in Cr)', 'Pred Value (in Cr)'], axis=1, inplace=True)

In [50]:
df['Month'] = (
    pd.to_datetime(df['Month']) + pd.offsets.MonthEnd(0)
)

In [51]:
df['run_month'] = pd.to_datetime(df['run_month'])
df['Month'] = pd.to_datetime(df['Month'])


mappings = {}

for run_month in df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-05-31 00:00:00'): {Timestamp('2026-05-31 00:00:00'): 'M',
  Timestamp('2026-06-30 00:00:00'): 'M+1',
  Timestamp('2026-07-31 00:00:00'): 'M+2',
  Timestamp('2026-08-31 00:00:00'): 'M+3',
  Timestamp('2026-09-30 00:00:00'): 'M+4',
  Timestamp('2026-10-31 00:00:00'): 'M+5',
  Timestamp('2026-11-30 00:00:00'): 'M+6',
  Timestamp('2026-12-31 00:00:00'): 'M+7',
  Timestamp('2027-01-31 00:00:00'): 'M+8'}}

In [52]:
df['M month'] = df.apply(
    lambda x: mappings[x['run_month']].get(
        x['Month']
    ), axis=1
)

In [53]:
df

,Channel,Sub Channel,Portfolio,Brand,Brand Class,Qtr Index Rate,Key,Month,ASM,Depot,...,P3M Val (Cr),P6M Val (Cr),LY P3M Val (Cr),LY P6M Val (Cr),LY P3M (Rolling) Val (Cr),Pred Val (Cr),Primary P3M 0?,NPD Flag,run_month,M month
0,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-06-30,BCE1,D231,...,0.000347,0.000517,0.000231,0.000231,0.000347,0.000593,False,NON-NPD,2026-05-31,M+1
1,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-07-31,BCE1,D231,...,0.000347,0.000517,0.000231,0.000289,0.000347,0.000417,False,NON-NPD,2026-05-31,M+2
2,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-08-31,BCE1,D231,...,0.000347,0.000517,0.000231,0.000289,0.000347,0.001111,False,NON-NPD,2026-05-31,M+3
3,MT,B2B,Others,REV.LQDST,B,247926.608903,BCE1_D231_718303,2026-09-30,BCE1,D231,...,0.000347,0.000517,0.000231,0.000405,0.000463,0.000000,False,NON-NPD,2026-05-31,M+4
4,MT,B2B,CNO,PCNO(R),A,349274.001420,BCE1_D231_718312,2026-06-30,BCE1,D231,...,0.002328,0.002212,0.000000,0.001164,0.000000,0.001929,False,NON-NPD,2026-05-31,M+1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26883,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811219,2026-09-30,MCW2,D463,...,0.000541,0.000000,0.000000,0.000000,0.000000,0.000541,False,NON-NPD,2026-05-31,M+4
26884,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-06-30,MCW2,D463,...,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31,M+1
26885,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-07-31,MCW2,D463,...,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31,M+2
26886,MT,B2C,Foods,TRU_ELMNT,NPD,353000.000000,MCW2_D463_811220,2026-08-31,MCW2,D463,...,0.000812,0.000000,0.000000,0.000000,0.000000,0.000812,False,NON-NPD,2026-05-31,M+3


In [55]:
df['Month'] = df['Month'].astype(str)
df['run_month'] = df['run_month'].astype(str)
df.columns

Index(['Channel', 'Sub Channel', 'Portfolio', 'Brand', 'Brand Class',
       'Qtr Index Rate', 'Key', 'Month', 'ASM', 'Depot', 'PSKU',
       'LY Vol (ROUM)', 'P3M Vol (ROUM)', 'P6M Vol (ROUM)',
       'LY P3M Vol (ROUM)', 'LY P6M Vol (ROUM)', 'LY P3M (Rolling) Vol (ROUM)',
       'Pred Vol (ROUM)', 'Sec Value Lag 1 (Cr)', 'Sec Value Lag 2 (Cr)',
       'Sec Value Lag 3 (Cr)', 'LY Value Lag 1 (Cr)', 'LY Value Lag 2 (Cr)',
       'LY Value Lead 1 (Cr)', 'LY Value Lead 2 (Cr)', 'Planning Principle',
       'LY Val (Cr)', 'P3M Val (Cr)', 'P6M Val (Cr)', 'LY P3M Val (Cr)',
       'LY P6M Val (Cr)', 'LY P3M (Rolling) Val (Cr)', 'Pred Val (Cr)',
       'Primary P3M 0?', 'NPD Flag', 'run_month', 'M month'],
      dtype='object')

In [56]:
upload_df = df[['Channel','Portfolio', 'Brand', 'Brand Class',
        'run_month', 'M month','Month', 'ASM', 'Depot', 'PSKU','Pred Vol (ROUM)']]

In [57]:
upload_df

,Channel,Portfolio,Brand,Brand Class,run_month,M month,Month,ASM,Depot,PSKU,Pred Vol (ROUM)
0,MT,Others,REV.LQDST,B,2026-05-31,M+1,2026-06-30,BCE1,D231,718303,0.023937
1,MT,Others,REV.LQDST,B,2026-05-31,M+2,2026-07-31,BCE1,D231,718303,0.016800
2,MT,Others,REV.LQDST,B,2026-05-31,M+3,2026-08-31,BCE1,D231,718303,0.044800
3,MT,Others,REV.LQDST,B,2026-05-31,M+4,2026-09-30,BCE1,D231,718303,0.000000
4,MT,CNO,PCNO(R),A,2026-05-31,M+1,2026-06-30,BCE1,D231,718312,0.055225
...,...,...,...,...,...,...,...,...,...,...,...
26883,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+4,2026-09-30,MCW2,D463,811219,0.015333
26884,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+1,2026-06-30,MCW2,D463,811220,0.023000
26885,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+2,2026-07-31,MCW2,D463,811220,0.023000
26886,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+3,2026-08-31,MCW2,D463,811220,0.023000


In [ ]:
# upload_df[upload_df['Month'] == '2026-04-30']['Pred Vol (ROUM)'].sum()

5346018.0036901245

In [58]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PRED VOL (ROUM)
0,MT,Others,REV.LQDST,B,2026-05-31,M+1,2026-06-30,BCE1,D231,718303,0.023937
1,MT,Others,REV.LQDST,B,2026-05-31,M+2,2026-07-31,BCE1,D231,718303,0.016800
2,MT,Others,REV.LQDST,B,2026-05-31,M+3,2026-08-31,BCE1,D231,718303,0.044800
3,MT,Others,REV.LQDST,B,2026-05-31,M+4,2026-09-30,BCE1,D231,718303,0.000000
4,MT,CNO,PCNO(R),A,2026-05-31,M+1,2026-06-30,BCE1,D231,718312,0.055225
...,...,...,...,...,...,...,...,...,...,...,...
26883,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+4,2026-09-30,MCW2,D463,811219,0.015333
26884,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+1,2026-06-30,MCW2,D463,811220,0.023000
26885,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+2,2026-07-31,MCW2,D463,811220,0.023000
26886,MT,Foods,TRU_ELMNT,NPD,2026-05-31,M+3,2026-08-31,MCW2,D463,811220,0.023000


In [59]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_SHARED",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 26888,
 [('xbckuyjjbo/file0.txt',
   'LOADED',
   26888,
   26888,
   1,
   0,
   None,
   None,
   None,
   None)])

### push offtakes to primary

In [5]:
qcom_df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Ecom_depot_psku_forecast_as_on_12th_may_2026.xlsx',
    sheet_name='Base'
)

In [6]:
qcom_df

,Key,Depot,PSKU,PSKU desciption,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,...,LY Primary Actuals Lead 1 Val,LY Primary Actuals Lead 2 Val,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val,M Month,Brand Class,NPD Flag,Planning Principle,Primary P3M 0?
0,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-06-30,NaN,0,0,...,0.0,0.0,0.0,0.000000,0.0,M+1,A,NON-NPD,Valid,True
1,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-07-31,NaN,0,0,...,0.0,0.0,0.0,0.000000,0.0,M+2,A,NON-NPD,Valid,True
2,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-08-31,NaN,0,0,...,0.0,0.0,0.0,0.000000,0.0,M+3,A,NON-NPD,Valid,True
3,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-09-30,NaN,0,0,...,0.0,0.0,0.0,0.000000,0.0,M+4,A,NON-NPD,Valid,True
4,D111_710981,D111,710981,PA Men 200ml Almond Hair Oil BT,PA_MEN_AH,2026-05-31,2026-06-30,NaN,0,0,...,0.0,0.0,0.0,0.000000,0.0,M+1,NaN,NON-NPD,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76907,D677_811269,D677,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,2026-05-31,2026-09-30,0.0,0,0,...,NaN,NaN,NaN,0.006158,0.0,M+4,NPD,NPD,Valid,True
76908,D677_811279,D677,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,2026-05-31,2026-06-30,0.0,0,0,...,NaN,NaN,NaN,0.000000,0.0,M+1,NPD,NON-NPD,Valid,True
76909,D677_811279,D677,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,2026-05-31,2026-07-31,0.0,0,0,...,NaN,NaN,NaN,0.000000,0.0,M+2,NPD,NON-NPD,Valid,True
76910,D677_811279,D677,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,2026-05-31,2026-08-31,0.0,0,0,...,NaN,NaN,NaN,0.000000,0.0,M+3,NPD,NON-NPD,Valid,True


In [ ]:
# qcom_df['Month Date'] = (
#     pd.to_datetime(qcom_df['Month Date'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# qcom_df

ValueError: '0       2026-05-31
1       2026-06-30
2       2026-07-31
3       2026-08-31
4       2026-05-31
           ...    
41751   2026-08-31
41752   2026-05-31
41753   2026-06-30
41754   2026-07-31
41755   2026-08-31
Name: Month Date, Length: 41756, dtype: datetime64[ns]' is not compatible with origin='1899-12-30'; it must be numeric with a unit specified

In [ ]:
# qcom_df['Month Date'] = (
#     pd.to_datetime(qcom_df['Month Date']) + pd.offsets.MonthEnd(0)
# )

In [ ]:
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month']) + pd.offsets.MonthEnd(0)
# )
# qcom_df

,Key,Depot,PSKU,PSKU Desc,Brand,Portfolio,Index Rate,Run Month,Month Date,M Month,...,LY Offtake Chain FC PSKU Lead 2 Val,Actual Closing SOH Val,Actual Closing SOH Lag 1 Val,Actual Closing SOH Lag 2 Val,Final Assumed Closing SOH Val,Final Assumed Closing SOH Lag 1 Val,Safety Stock Val,Brand Class,Planning Principle,Primary P3M 0?
0,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-03-31,M+1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
1,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-04-30,M+2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
2,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-05-31,M+3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
3,D112_709567,D112,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,Foods,126480.737807,2026-02-28,2026-06-30,M+4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,Valid,True
4,D112_718287,D112,718287,PCNO 200ml JAR,PCNO(R),CNO,309765.865129,2026-02-28,2026-03-31,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A,Valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30603,D677_810805,D677,810805,PA BABY FACE BODY WIPES 1KG,PABABY_GM,Skin Care,451.133000,2026-02-28,2026-06-30,M+4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30604,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-03-31,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30605,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-04-30,M+2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True
30606,D677_811181,D677,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,Saffola Oils,330000.000000,2026-02-28,2026-05-31,M+3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,C,Valid,True


In [7]:
qcom_df['Run Month'] = pd.to_datetime(qcom_df['Run Month'])
qcom_df['Month Date'] = pd.to_datetime(qcom_df['Month Date'])

mappings ={}

for run_month in qcom_df['Run Month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


   
qcom_df['M month'] = qcom_df.apply(
    lambda x: mappings[x['Run Month']].get(
        x['Month Date']
    ), axis=1
)

In [8]:
qcom_df.columns

Index(['Key', 'Depot', 'PSKU', 'PSKU desciption', 'Brand', 'Run Month',
       'Month Date', 'Calculated PSKU Primary Vol', 'Primary Actuals Vol',
       'Primary Plan Vol', 'Secondary Plan Vol', 'Secondary Actuals Vol',
       'Primary P3M Vol', 'Primary Actuals Lag 1 Vol',
       'Primary Actuals Lag 2 Vol', 'Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actuals Lead 2 Vol',
       'LY Primary P3M Vol', 'Primary P3M copy Vol',
       'PSKU Primary P3M Sum Vol', 'PSKU P3M Contribution',
       'Calculated Depot PSKU Primary Vol', 'Index Rate',
       'Calculated PSKU Primary Val', 'Primary Actuals Val',
       'Primary Plan Val', 'Secondary Plan Val', 'Secondary Actuals Val',
       'Primary P3M Val', 'Primary Actuals Lag 1 Val',
       'Primary Actuals Lag 2 Val', 'Primary Actuals Lag 3 Val',
       'LY Primary Actua

In [9]:
qcom_df.rename(columns = {'Calculated Depot PSKU Primary Vol':'Calculated Primary Vol'}, inplace = True)

In [10]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [11]:

len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    brand_md_df,
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(qcom_df)

In [12]:
qcom_df

,Key,Depot,PSKU,PSKU desciption,Brand,Run Month,Month Date,Calculated PSKU Primary Vol,Primary Actuals Vol,Primary Plan Vol,...,LY Primary P3M Val,PSKU Primary P3M Sum Val,Calculated Depot PSKU Primary Val,M Month,Brand Class,NPD Flag,Planning Principle,Primary P3M 0?,M month,portfolio
0,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-06-30,NaN,0,0,...,0.0,0.000000,0.0,M+1,A,NON-NPD,Valid,True,M+1,Foods
1,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-07-31,NaN,0,0,...,0.0,0.000000,0.0,M+2,A,NON-NPD,Valid,True,M+2,Foods
2,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-08-31,NaN,0,0,...,0.0,0.000000,0.0,M+3,A,NON-NPD,Valid,True,M+3,Foods
3,D111_709567,D111,709567,SAF OATS 400G(FRE WT SAF OATS 1KG P)-N,SAFF OATS,2026-05-31,2026-09-30,NaN,0,0,...,0.0,0.000000,0.0,M+4,A,NON-NPD,Valid,True,M+4,Foods
4,D111_710981,D111,710981,PA Men 200ml Almond Hair Oil BT,PA_MEN_AH,2026-05-31,2026-06-30,NaN,0,0,...,0.0,0.000000,0.0,M+1,NaN,NON-NPD,Valid,True,M+1,Hair Oils
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76907,D677_811269,D677,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,2026-05-31,2026-09-30,0.0,0,0,...,NaN,0.006158,0.0,M+4,NPD,NPD,Valid,True,M+4,Hair Oils
76908,D677_811279,D677,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,2026-05-31,2026-06-30,0.0,0,0,...,NaN,0.000000,0.0,M+1,NPD,NON-NPD,Valid,True,M+1,Saffola Oils
76909,D677_811279,D677,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,2026-05-31,2026-07-31,0.0,0,0,...,NaN,0.000000,0.0,M+2,NPD,NON-NPD,Valid,True,M+2,Saffola Oils
76910,D677_811279,D677,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,2026-05-31,2026-08-31,0.0,0,0,...,NaN,0.000000,0.0,M+3,NPD,NON-NPD,Valid,True,M+3,Saffola Oils


In [13]:
qcom_df = qcom_df.groupby(['Month Date', 'Depot', 'PSKU',
       'Brand', 'portfolio','M month','Run Month'])[['Calculated Primary Vol']].sum().reset_index()
qcom_df

,Month Date,Depot,PSKU,Brand,portfolio,M month,Run Month,Calculated Primary Vol
0,2026-06-30,D111,709567,SAFF OATS,Foods,M+1,2026-05-31,0.0
1,2026-06-30,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-05-31,0.0
2,2026-06-30,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
3,2026-06-30,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
4,2026-06-30,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
...,...,...,...,...,...,...,...,...
73855,2026-09-30,D677,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0
73856,2026-09-30,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
73857,2026-09-30,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
73858,2026-09-30,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0


In [14]:

qcom_df.rename(columns = {'Month Date':'Month', 'Final PSKU':'PSKU', 'Depot Code':'Depot', 'Run Month':'run_month','portfolio':'Portfolio'}, inplace = True)
qcom_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-06-30,D111,709567,SAFF OATS,Foods,M+1,2026-05-31,0.0
1,2026-06-30,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-05-31,0.0
2,2026-06-30,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
3,2026-06-30,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
4,2026-06-30,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
...,...,...,...,...,...,...,...,...
73855,2026-09-30,D677,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0
73856,2026-09-30,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
73857,2026-09-30,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
73858,2026-09-30,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0


In [15]:
qcom_df[qcom_df['Month'] == '2026-06-30']['Calculated Primary Vol'].sum()

285627.78546275187

In [16]:
qcom_df['Month'] = qcom_df['Month'].astype(str)
qcom_df['run_month'] = qcom_df['run_month'].astype(str)
qcom_df.columns

Index(['Month', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol'],
      dtype='object')

In [17]:
qcom_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-06-30,D111,709567,SAFF OATS,Foods,M+1,2026-05-31,0.0
1,2026-06-30,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-05-31,0.0
2,2026-06-30,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
3,2026-06-30,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
4,2026-06-30,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0
...,...,...,...,...,...,...,...,...
73855,2026-09-30,D677,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0
73856,2026-09-30,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
73857,2026-09-30,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
73858,2026-09-30,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0


In [18]:
upload_df = qcom_df[['Month', 'Depot', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol']]


In [19]:
upload_df['Channel'] = 'ECOM'

In [20]:
upload_df

,Month,Depot,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol,Channel
0,2026-06-30,D111,709567,SAFF OATS,Foods,M+1,2026-05-31,0.0,ECOM
1,2026-06-30,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-05-31,0.0,ECOM
2,2026-06-30,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0,ECOM
3,2026-06-30,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0,ECOM
4,2026-06-30,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0,ECOM
...,...,...,...,...,...,...,...,...,...
73855,2026-09-30,D677,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0,ECOM
73856,2026-09-30,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,ECOM
73857,2026-09-30,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,ECOM
73858,2026-09-30,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,ECOM


In [21]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH,DEPOT,PSKU,BRAND,PORTFOLIO,M MONTH,RUN_MONTH,CALCULATED PRIMARY VOL,CHANNEL
0,2026-06-30,D111,709567,SAFF OATS,Foods,M+1,2026-05-31,0.0,ECOM
1,2026-06-30,D111,710981,PA_MEN_AH,Hair Oils,M+1,2026-05-31,0.0,ECOM
2,2026-06-30,D111,711040,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0,ECOM
3,2026-06-30,D111,711041,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0,ECOM
4,2026-06-30,D111,711042,PCNO GOLD,Hair Oils,M+1,2026-05-31,0.0,ECOM
...,...,...,...,...,...,...,...,...,...
73855,2026-09-30,D677,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0,ECOM
73856,2026-09-30,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,ECOM
73857,2026-09-30,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,ECOM
73858,2026-09-30,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,ECOM


In [154]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFT2PRIM_SHARED",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 73860,
 [('stcdmvujny/file0.txt',
   'LOADED',
   73860,
   73860,
   1,
   0,
   None,
   None,
   None,
   None)])

### Push chain psku primary

In [68]:
qcom_df = pd.read_excel(
    '/data/aman_singh/acuuracy_check/Ecom_chain_psku_forecast_as_on_12th_may_2026.xlsx',
    sheet_name='Base'
)

In [69]:
qcom_df

,Key,Chain,PSKU,PSKU description,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand class,NPD Flag,Planning Principle,Primary P3M 0?
0,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-06-30,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0
1,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-07-31,M+2,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0
2,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-08-31,M+3,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0
3,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-09-30,M+4,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0
4,Amazon ARIPL_715099,Amazon ARIPL,715099,COCOSOUL FOOT CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-06-30,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25755,Purplle_811269,Purplle,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-05-31,2026-09-30,M+4,...,0.0,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0
25756,Purplle_811279,Purplle,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-05-31,2026-06-30,M+1,...,0.0,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0
25757,Purplle_811279,Purplle,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-05-31,2026-07-31,M+2,...,0.0,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0
25758,Purplle_811279,Purplle,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-05-31,2026-08-31,M+3,...,0.0,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0


In [ ]:
# qcom_df['Month Date'] = (
#     pd.to_datetime(qcom_df['Month Date'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# qcom_df

ValueError: '0       2026-05-31
1       2026-06-30
2       2026-07-31
3       2026-08-31
4       2026-05-31
           ...    
73471   2026-08-31
73472   2026-05-31
73473   2026-06-30
73474   2026-07-31
73475   2026-08-31
Name: Month Date, Length: 73476, dtype: datetime64[ns]' is not compatible with origin='1899-12-30'; it must be numeric with a unit specified

In [ ]:
# qcom_df['Month Date'] = (
#     pd.to_datetime(qcom_df['Month Date']) + pd.offsets.MonthEnd(0)
# )

In [ ]:
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# qcom_df['Run Month'] = (
#     pd.to_datetime(qcom_df['Run Month']) + pd.offsets.MonthEnd(0)
# )
# qcom_df

,Key,Chain,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,Offtake Actuals Lag 3 Val,LY Offtake Actuals Lag 1 Val,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand Class,Planning Principle,Primary P3M 0?
0,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-03-31,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
1,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-04-30,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
2,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-05-31,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
3,Amazon ARIPL_718287,Amazon ARIPL,718287,PCNO 200ml JAR,PCNO(R),309765.865129,CNO,2026-02-28,2026-06-30,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,A,Valid,True
4,Amazon ARIPL_718288,Amazon ARIPL,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.28265,0.165361,0.190966,0.22004,0.159827,0.200795,0.34632,A,Valid,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11487,Nykaa_811169,Nykaa,811169,SW MINI EDP GIFT PO4 18ML,SW_SGPRF,1443.400363,Male Grooming,2026-02-28,2026-06-30,M+4,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11488,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-03-31,M+1,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11489,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-04-30,M+2,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True
11490,Nykaa_811181,Nykaa,811181,SAFFOLA COLDPRESS CNO 1L,SAF_CDPRS,330000.000000,Saffola Oils,2026-02-28,2026-05-31,M+3,...,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,C,Valid,True


In [70]:
qcom_df['Run Month'] = pd.to_datetime(qcom_df['Run Month'])
qcom_df['Month Date'] = pd.to_datetime(qcom_df['Month Date'])

mappings ={}

for run_month in qcom_df['Run Month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


   
qcom_df['M month'] = qcom_df.apply(
    lambda x: mappings[x['Run Month']].get(
        x['Month Date']
    ), axis=1
)

In [ ]:
# qcom_df = qcom_df[qcom_df['M month'] == 'M+1']
# qcom_df

,key2,key,Chain,FC,PSKU,PSKU Desc,Brand,Index Rate,Portfolio,Run Month,...,Offtake Chain FC PSKU Lag 3 Vol,LY Offtake Actuals Chain FC PSKU Vol,LY Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU P3M Vol,Offtake Chain FC PSKU Lag 1 Val,Offtake Chain FC PSKU Lag 2 Val,Offtake Chain FC PSKU Lag 3 Val,LY Offtake Actuals Chain FC PSKU Val,LY Offtake Chain FC PSKU P3M Val,Offtake Chain FC PSKU P3M Val
2,Blinkit_ahmedabad a2 - feeder warehouse_718288,Blinkit_718288,Blinkit,ahmedabad a2 - feeder warehouse,718288,SAFF GOLD 5L JAR,SAFF GOLD,137662.938527,Saffola Oils,2025-11-30,...,6.873931,5.219777,3.804556,6.602009,0.107562,0.085510,0.094629,0.071857,0.052375,0.090885
9,Blinkit_ahmedabad a2 - feeder warehouse_718299,Blinkit_718299,Blinkit,ahmedabad a2 - feeder warehouse,718299,PCNO 100ml BTL,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
16,Blinkit_ahmedabad a2 - feeder warehouse_718308,Blinkit_718308,Blinkit,ahmedabad a2 - feeder warehouse,718308,PCNO 100ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
23,Blinkit_ahmedabad a2 - feeder warehouse_718310,Blinkit_718310,Blinkit,ahmedabad a2 - feeder warehouse,718310,PCNO 500ml JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30,Blinkit_ahmedabad a2 - feeder warehouse_718312,Blinkit_718312,Blinkit,ahmedabad a2 - feeder warehouse,718312,PCNO 1L JAR,PCNO(R),309765.865129,CNO,2025-11-30,...,0.196489,0.114393,0.065902,0.201895,0.006292,0.005917,0.006087,0.003543,0.002041,0.006254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235825,Zepto_pun-dry-mh2-koregaon_810673,Zepto_810673,Zepto,pun-dry-mh2-koregaon,810673,PA ESS ROSEMARY OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235832,Zepto_pun-dry-mh2-koregaon_810674,Zepto_810674,Zepto,pun-dry-mh2-koregaon,810674,PA ESS TEA TREE OIL 14ML,PA_ESS_HO,12900.000000,Hair Oils,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235839,Zepto_pun-dry-mh2-koregaon_810685,Zepto_810685,Zepto,pun-dry-mh2-koregaon,810685,SF MUESLI MANGO 400G POUCH,SAF-MUSLI,321959.667548,Foods,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
235846,Zepto_pun-dry-mh2-koregaon_810738,Zepto_810738,Zepto,pun-dry-mh2-koregaon,810738,PA BABY FACE BODY WIPE 362GM,PABABY_GM,451.133000,Skin Care,2025-11-30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [71]:
qcom_df.columns

Index(['Key', 'Chain', 'PSKU', 'PSKU description', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date', 'M month',
       'Primary Till Date Actuals Vol', 'Secondary Plan Vol',
       'Primary P3M Vol', 'Offtake Chain PSKU Vol',
       'Offtake Chain PSKU Forecast Vol', 'Norms SOH', 'Norm Days',
       'Safety Stock Vol', 'Actual Closing SOH Vol',
       'Actual Closing SOH Lag 1 Vol', 'Actual Closing SOH Lag 2 Vol',
       'Assumed Closing SOH Vol', 'Assumed Closing SOH Lag 1 Vol',
       'Final Assumed Closing SOH Vol', 'Final Assumed Closing SOH Lag 1 Vol',
       'Primary Actuals Vol', 'Sec Actuals Vol', 'Primary P3M redundant Vol',
       'LY Primary Actuals Vol', 'LY Sec Actuals Vol', 'LY Primary P3M Vol',
       'Primary Actuals Lag 1 Vol', 'Primary Actuals Lag 2 Vol',
       'Primary Actuals Lag 3 Vol', 'LY Primary Actuals Lag 1 Vol',
       'LY Primary Actuals Lag 2 Vol', 'LY Primary Actuals Lag 3 Vol',
       'LY Primary Actuals Lead 1 Vol', 'LY Primary Actua

In [72]:
qcom_df.rename(columns = {'Calculated Depot PSKU Primary Vol':'Calculated Primary Vol'}, inplace = True)

In [73]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [74]:

len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    brand_md_df,
    on=['Brand'],
    how='left'
)
assert len_before_merge == len(qcom_df)

In [75]:
qcom_df

,Key,Chain,PSKU,PSKU description,Brand,Index Rate,Portfolio,Run Month,Month Date,M month,...,LY Offtake Actuals Lag 2 Val,LY Offtake Actuals Lag 3 Val,LY Offtake Actuals Lead 1 Val,LY Offtake Actuals Lead 2 Val,Calculated Primary Val,Brand class,NPD Flag,Planning Principle,Primary P3M 0?,portfolio
0,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-06-30,M+1,...,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0,Skin Care
1,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-07-31,M+2,...,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0,Skin Care
2,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-08-31,M+3,...,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0,Skin Care
3,Amazon ARIPL_715098,Amazon ARIPL,715098,COCOSOUL H&N CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-09-30,M+4,...,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0,Skin Care
4,Amazon ARIPL_715099,Amazon ARIPL,715099,COCOSOUL FOOT CREAM 75ML,CO_SO_PCP,1220.081000,Skin Care,2026-05-31,2026-06-30,M+1,...,0.0,0.0,0.0,0.0,0.0,NaN,NON-NPD,Valid,1.0,Skin Care
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25755,Purplle_811269,Purplle,811269,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,12860.631072,Hair Oils,2026-05-31,2026-09-30,M+4,...,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0,Hair Oils
25756,Purplle_811279,Purplle,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-05-31,2026-06-30,M+1,...,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0,Saffola Oils
25757,Purplle_811279,Purplle,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-05-31,2026-07-31,M+2,...,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0,Saffola Oils
25758,Purplle_811279,Purplle,811279,SAFFOLA COLDPRESS SSO GROUNDNUT 2L,SAF_CDPRS,260000.000000,Saffola Oils,2026-05-31,2026-08-31,M+3,...,0.0,0.0,0.0,0.0,0.0,NPD,NON-NPD,Valid,1.0,Saffola Oils


In [76]:
qcom_df = qcom_df.groupby(['Month Date', 'Chain', 'PSKU',
       'Brand', 'portfolio','M month','Run Month'])[['Calculated Primary Vol']].sum().reset_index()
qcom_df

,Month Date,Chain,PSKU,Brand,portfolio,M month,Run Month,Calculated Primary Vol
0,2026-06-30,Amazon ARIPL,715098,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
1,2026-06-30,Amazon ARIPL,715099,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
2,2026-06-30,Amazon ARIPL,715100,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
3,2026-06-30,Amazon ARIPL,715106,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
4,2026-06-30,Amazon ARIPL,715107,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
...,...,...,...,...,...,...,...,...
24643,2026-09-30,Purplle,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0
24644,2026-09-30,Purplle,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
24645,2026-09-30,Purplle,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
24646,2026-09-30,Purplle,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0


In [77]:

qcom_df.rename(columns = {'Month Date':'Month', 'Final PSKU':'PSKU', 'Depot Code':'Depot', 'Run Month':'run_month','portfolio':'Portfolio'}, inplace = True)
qcom_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-06-30,Amazon ARIPL,715098,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
1,2026-06-30,Amazon ARIPL,715099,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
2,2026-06-30,Amazon ARIPL,715100,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
3,2026-06-30,Amazon ARIPL,715106,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
4,2026-06-30,Amazon ARIPL,715107,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
...,...,...,...,...,...,...,...,...
24643,2026-09-30,Purplle,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0
24644,2026-09-30,Purplle,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
24645,2026-09-30,Purplle,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
24646,2026-09-30,Purplle,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0


In [78]:
qcom_df[qcom_df['Month'] == '2026-06-30']['Calculated Primary Vol'].sum()

291909.7466504647

In [79]:
qcom_df['Month'] = qcom_df['Month'].astype(str)
qcom_df['run_month'] = qcom_df['run_month'].astype(str)
qcom_df.columns

Index(['Month', 'Chain', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol'],
      dtype='object')

In [80]:
qcom_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol
0,2026-06-30,Amazon ARIPL,715098,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
1,2026-06-30,Amazon ARIPL,715099,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
2,2026-06-30,Amazon ARIPL,715100,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
3,2026-06-30,Amazon ARIPL,715106,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
4,2026-06-30,Amazon ARIPL,715107,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0
...,...,...,...,...,...,...,...,...
24643,2026-09-30,Purplle,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0
24644,2026-09-30,Purplle,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
24645,2026-09-30,Purplle,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0
24646,2026-09-30,Purplle,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0


In [81]:
upload_df = qcom_df[['Month', 'Chain', 'PSKU', 'Brand', 'Portfolio', 'M month', 'run_month',
       'Calculated Primary Vol']]


In [82]:
upload_df['Channel'] = 'Ecom'

In [83]:
upload_df

,Month,Chain,PSKU,Brand,Portfolio,M month,run_month,Calculated Primary Vol,Channel
0,2026-06-30,Amazon ARIPL,715098,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
1,2026-06-30,Amazon ARIPL,715099,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
2,2026-06-30,Amazon ARIPL,715100,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
3,2026-06-30,Amazon ARIPL,715106,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
4,2026-06-30,Amazon ARIPL,715107,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
...,...,...,...,...,...,...,...,...,...
24643,2026-09-30,Purplle,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0,Ecom
24644,2026-09-30,Purplle,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,Ecom
24645,2026-09-30,Purplle,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,Ecom
24646,2026-09-30,Purplle,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,Ecom


In [84]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH,CHAIN,PSKU,BRAND,PORTFOLIO,M MONTH,RUN_MONTH,CALCULATED PRIMARY VOL,CHANNEL
0,2026-06-30,Amazon ARIPL,715098,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
1,2026-06-30,Amazon ARIPL,715099,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
2,2026-06-30,Amazon ARIPL,715100,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
3,2026-06-30,Amazon ARIPL,715106,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
4,2026-06-30,Amazon ARIPL,715107,CO_SO_PCP,Skin Care,M+1,2026-05-31,0.0,Ecom
...,...,...,...,...,...,...,...,...,...
24643,2026-09-30,Purplle,811181,SAF_CDPRS,Saffola Oils,M+4,2026-05-31,0.0,Ecom
24644,2026-09-30,Purplle,811267,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,Ecom
24645,2026-09-30,Purplle,811268,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,Ecom
24646,2026-09-30,Purplle,811269,PA_ESS_HO,Hair Oils,M+4,2026-05-31,0.0,Ecom


In [85]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFT2PRIM_CPSKU",
            auto_create_table=True,
            overwrite = False,)

(True,
 1,
 24648,
 [('gnzcsweect/file0.txt',
   'LOADED',
   24648,
   24648,
   1,
   0,
   None,
   None,
   None,
   None)])

### push offtakes data

In [37]:
offtakes_df = pd.read_excel('/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_cp_may_live.xlsx', sheet_name = 'Base')

In [38]:
offtakes_df.columns[50:]

Index(['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3',
       'class', 'skipped', 'seasonality_flag', 'final_trend',
       'lower_threshold', 'upper_threshold', 'recency_factor',
       'shrink_ratio_prophet', 'shrink_ratio_rf', 'p3m_ly_growth',
       'recency_heuristic_prophet_vol', 'seasonal_heuristic_prophet_vol',
       'rec_seas_heuristic_prophet_vol', 'non_seasonal_heuristic_prophet_vol',
       'final_heuristic_prophet_vol', 'final_heuristic_prophet_value',
       'final_heuristic_prophet_value_2', 'final_heuristic_prophet_vol_2',
       'recency_heuristic_rf_vol', 'seasonal_heuristic_rf_vol',
       'rec_seas_heuristic_rf_vol', 'non_seasonal_heuristic_rf_vol',
       'final_heuristic_rf_vol', 'final_heuristic_rf_value',
       'final_heuristic_rf_value_2', 'final_heuristic_rf_vol_2',
       'error_prophet_vol', 'abs_error_prophet_vol', 'error_prophet_value',
       'abs_error_prophet_value', 'error_rf_vol', 'abs_error_rf_vol',
       'error_rf_value', 

In [ ]:
offtakes_df.rename(columns = {'run_month_x':'run_month'}, inplace = True)
offtakes_df['run_month'].unique()

<DatetimeArray>
['2026-05-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [41]:
offtakes_df = offtakes_df.groupby(['month_date','platform_name', 'parent_material_code', 'brand_code','portfolio','run_month', 'M month']
                    )[['pred_prophet', 'pred_rf','final_heuristic_value']].sum().reset_index()

In [42]:
offtakes_df.rename(columns = {'final_heuristic_value':'final_heuristic_prophet_value_2'}, inplace = True)

In [43]:
offtakes_df['month_date'] = offtakes_df['month_date'].astype(str)
offtakes_df['run_month'] = offtakes_df['run_month'].astype(str)
offtakes_df.columns

Index(['month_date', 'platform_name', 'parent_material_code', 'brand_code',
       'portfolio', 'run_month', 'M month', 'pred_prophet', 'pred_rf',
       'final_heuristic_prophet_value_2'],
      dtype='object')

In [44]:
upload_df = offtakes_df.copy()

In [45]:
upload_df

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,M month,pred_prophet,pred_rf,final_heuristic_prophet_value_2
0,2026-05-31,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-05-31,M,56.509207,60.788678,0.847068
1,2026-05-31,Blinkit,718310,PCNO(R),CNO,2026-05-31,M,0.067486,0.068389,0.000096
2,2026-05-31,Blinkit,718312,PCNO(R),CNO,2026-05-31,M,7.828529,6.195487,0.281297
3,2026-05-31,Blinkit,718315,PCNO(R),CNO,2026-05-31,M,0.000000,0.000000,0.000000
4,2026-05-31,Blinkit,718317,H&C,Hair Oils,2026-05-31,M,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
7147,2026-12-31,Zepto,810685,SAF-MUSLI,Foods,2026-05-31,M+7,0.000000,0.000000,0.000194
7148,2026-12-31,Zepto,810738,PABABY_GM,Skin Care,2026-05-31,M+7,0.000000,0.000000,0.005072
7149,2026-12-31,Zepto,810971,PA_ESS_HO,Hair Oils,2026-05-31,M+7,0.000000,0.000000,0.000752
7150,2026-12-31,Zepto,811005,PA_ESS_HO,Hair Oils,2026-05-31,M+7,0.000000,0.000000,0.000630


In [46]:
upload_df['channel'] = 'QCOM'

In [47]:
upload_df.columns = upload_df.columns.str.upper()
upload_df

,MONTH_DATE,PLATFORM_NAME,PARENT_MATERIAL_CODE,BRAND_CODE,PORTFOLIO,RUN_MONTH,M MONTH,PRED_PROPHET,PRED_RF,FINAL_HEURISTIC_PROPHET_VALUE_2,CHANNEL
0,2026-05-31,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-05-31,M,56.509207,60.788678,0.847068,QCOM
1,2026-05-31,Blinkit,718310,PCNO(R),CNO,2026-05-31,M,0.067486,0.068389,0.000096,QCOM
2,2026-05-31,Blinkit,718312,PCNO(R),CNO,2026-05-31,M,7.828529,6.195487,0.281297,QCOM
3,2026-05-31,Blinkit,718315,PCNO(R),CNO,2026-05-31,M,0.000000,0.000000,0.000000,QCOM
4,2026-05-31,Blinkit,718317,H&C,Hair Oils,2026-05-31,M,0.000000,0.000000,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
7147,2026-12-31,Zepto,810685,SAF-MUSLI,Foods,2026-05-31,M+7,0.000000,0.000000,0.000194,QCOM
7148,2026-12-31,Zepto,810738,PABABY_GM,Skin Care,2026-05-31,M+7,0.000000,0.000000,0.005072,QCOM
7149,2026-12-31,Zepto,810971,PA_ESS_HO,Hair Oils,2026-05-31,M+7,0.000000,0.000000,0.000752,QCOM
7150,2026-12-31,Zepto,811005,PA_ESS_HO,Hair Oils,2026-05-31,M+7,0.000000,0.000000,0.000630,QCOM


In [48]:
upload_df[upload_df['MONTH_DATE'] == '2026-06-30']['FINAL_HEURISTIC_PROPHET_VALUE_2'].sum()

32.73611362867541

In [49]:
# push data to snowflake
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, upload_df, 
            table_name = "TRN_MIL_DF_OFFTAKES_OUTPUT",
            auto_create_table=True,
            overwrite = False)

(True,
 1,
 7152,
 [('shhscliybm/file0.txt',
   'LOADED',
   7152,
   7152,
   1,
   0,
   None,
   None,
   None,
   None)])